In [1]:
# ==========================================
# BLOCKS 1 & 2: ARMORED PATHING & TYPE CASTING
# ==========================================
import pandas as pd
import os

print("Configuring paths and loading financial datasets...")
data_path = "../data"

# Safe pathing configuration using os.path.join
orders_file = os.path.join(data_path, "olist_orders_dataset.csv")
items_file = os.path.join(data_path, "olist_order_items_dataset.csv")
payments_file = os.path.join(data_path, "olist_order_payments_dataset.csv")
customers_file = os.path.join(data_path, "olist_customers_dataset.csv")

# Load datasets
orders = pd.read_csv(orders_file)
items = pd.read_csv(items_file)
payments = pd.read_csv(payments_file)
customers = pd.read_csv(customers_file)

# 1. STRICT ID TYPE CASTING (Preventing merge crashes across different data sources)
orders["order_id"] = orders["order_id"].astype(str)
orders["customer_id"] = orders["customer_id"].astype(str)
items["order_id"] = items["order_id"].astype(str)
payments["order_id"] = payments["order_id"].astype(str)
customers["customer_id"] = customers["customer_id"].astype(str)
customers["customer_unique_id"] = customers["customer_unique_id"].astype(str)

# 2. MONETARY NULL DEFENSE (Preventing calculation drops)
# If price or payment value is missing in raw input, safely default to 0.0 instead of throwing NaN errors
payments["payment_value"] = payments["payment_value"].fillna(0.0)
items["price"] = items["price"].fillna(0.0)
items["freight_value"] = items["freight_value"].fillna(0.0)

print("Financial datasets armored, type-cast, and validated successfully!")

Configuring paths and loading financial datasets...
Financial datasets armored, type-cast, and validated successfully!


In [2]:
# ==========================================
# BLOCK 2.5: ALL-TABLE NULL & DUPLICATE ANNIHILATION
# ==========================================
print("--- RUNNING FULL FINANCIAL INTEGRITY AUDIT ---")

# 1. Null Annihilation across ALL four core tables
tables = {
    "Orders": (orders, ['order_id', 'customer_id', 'order_status']),
    "Items": (items, ['order_id', 'product_id', 'price']),
    "Payments": (payments, ['order_id', 'payment_value']),
    "Customers": (customers, ['customer_id', 'customer_unique_id'])
}

for name, (df, keys) in tables.items():
    initial_len = len(df)
    df.dropna(subset=keys, inplace=True)
    dropped_nulls = initial_len - len(df)
    if dropped_nulls > 0:
        print(f"-> Annihilated {dropped_nulls} null rows from {name}")

# 2. Strict Deduplication for ALL Tables (No loose ends)
init_orders, init_cust = len(orders), len(customers)
init_items, init_pay = len(items), len(payments)

# Header tables: ID must be strictly unique
orders.drop_duplicates(subset=['order_id'], inplace=True)
customers.drop_duplicates(subset=['customer_id'], inplace=True)

# Line Item tables: We drop EXACT row duplicates to prevent double-charging
items.drop_duplicates(inplace=True)
payments.drop_duplicates(inplace=True)

print(f"-> Dropped {init_orders - len(orders)} duplicate orders.")
print(f"-> Dropped {init_cust - len(customers)} duplicate customers.")
print(f"-> Dropped {init_items - len(items)} duplicate items.")
print(f"-> Dropped {init_pay - len(payments)} duplicate payments.")

# 3. Anomaly Defense (Catching corrupt negative currency)
negative_payments = (payments['payment_value'] < 0).sum()
negative_prices = (items['price'] < 0).sum()

if negative_payments > 0 or negative_prices > 0:
    print(f"WARNING: Found {negative_payments} negative payments and {negative_prices} negative prices. Purging them.")
    payments = payments[payments['payment_value'] >= 0]
    items = items[items['price'] >= 0]
else:
    print("-> Financial anomaly check passed: No negative money values found.")

print("All financial tables are fully scrubbed and armored.")

--- RUNNING FULL FINANCIAL INTEGRITY AUDIT ---
-> Dropped 0 duplicate orders.
-> Dropped 0 duplicate customers.
-> Dropped 0 duplicate items.
-> Dropped 0 duplicate payments.
-> Financial anomaly check passed: No negative money values found.
All financial tables are fully scrubbed and armored.


In [3]:
print("All datasets loaded successfully!")
print("Orders:", orders.shape)
print("Items:", items.shape)
print("Payments:", payments.shape)
print("Customers:", customers.shape)

All datasets loaded successfully!
Orders: (99441, 8)
Items: (112650, 7)
Payments: (103886, 5)
Customers: (99441, 5)


In [4]:
print("Orders columns:")
print(orders.columns.tolist())

print("\nItems columns:")
print(items.columns.tolist())

print("\nPayments columns:")
print(payments.columns.tolist())

print("\nCustomers columns:")
print(customers.columns.tolist())

Orders columns:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

Items columns:
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

Payments columns:
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

Customers columns:
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']


In [5]:
delivered_orders = orders[
    orders["order_status"].eq("delivered")
].copy()

print("Delivered orders:", delivered_orders.shape)
print("Order statuses remaining:")
print(delivered_orders["order_status"].value_counts())

Delivered orders: (96478, 8)
Order statuses remaining:
order_status
delivered    96478
Name: count, dtype: int64


In [6]:
delivered_items = items[
    items["order_id"].isin(delivered_orders["order_id"])
].copy()

print("Items belonging to delivered orders:", delivered_items.shape)

Items belonging to delivered orders: (110197, 7)


In [7]:
items_total = (
    delivered_items
    .groupby("order_id", as_index=False)
    .agg(
        item_total=("price", "sum"),
        freight_total=("freight_value", "sum")
    )
)

print(items_total.head())
print("Item-level aggregation shape:", items_total.shape)

                           order_id  item_total  freight_total
0  00010242fe8c5a6d1ba2dd792cb16214       58.90          13.29
1  00018f77f2f0320c557190d7a144bdd3      239.90          19.93
2  000229ec398224ef6ca0657da4fc703e      199.00          17.87
3  00024acbcdf0a6daa1e931b038114c75       12.99          12.79
4  00042b26cf59d7ce69dfabb4e55b4fd9      199.90          18.14
Item-level aggregation shape: (96478, 3)


In [8]:
delivered_payments = payments[
    payments["order_id"].isin(delivered_orders["order_id"])
].copy()

print("Payments belonging to delivered orders: ", delivered_payments.shape)

payments_total = (
    delivered_payments
    .groupby("order_id", as_index=False)
    .agg(
        total_payment=("payment_value", "sum")
    )
)

print(payments_total.head())
print("Payment aggregation shape:", payments_total.shape)

Payments belonging to delivered orders:  (100756, 5)
                           order_id  total_payment
0  00010242fe8c5a6d1ba2dd792cb16214          72.19
1  00018f77f2f0320c557190d7a144bdd3         259.83
2  000229ec398224ef6ca0657da4fc703e         216.87
3  00024acbcdf0a6daa1e931b038114c75          25.78
4  00042b26cf59d7ce69dfabb4e55b4fd9         218.04
Payment aggregation shape: (96477, 2)


In [9]:
# ==========================================
# SAFE FINANCIAL MERGE
# ==========================================
order_financials = items_total.merge(
    payments_total,
    on="order_id",
    how="outer" # OUTER join prevents losing revenue if item records are missing
)

# Fill the gaps created by the outer join with 0 so math doesn't crash
order_financials["item_total"] = order_financials["item_total"].fillna(0)
order_financials["freight_total"] = order_financials["freight_total"].fillna(0)
order_financials["total_payment"] = order_financials["total_payment"].fillna(0)

print("Order financials shape:", order_financials.shape)
display(order_financials.head(3))

Order financials shape: (96478, 4)


,order_id,item_total,freight_total,total_payment
0,00010242fe8c5a6d1ba2dd792cb16214,58.9,13.29,72.19
1,00018f77f2f0320c557190d7a144bdd3,239.9,19.93,259.83
2,000229ec398224ef6ca0657da4fc703e,199.0,17.87,216.87


In [10]:
order_customer = delivered_orders[
    ["order_id", "customer_id"]
].merge(
    customers[
        ["customer_id", "customer_unique_id"]
    ],
    on="customer_id",
    how="left"
)

order_customer.head()

,order_id,customer_id,customer_unique_id
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,7c396fd4830fd04220f754e42b4e5bff
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,af07308b275d755c9edb36a90c618231
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,3a653a41f6f9fc3d2a113cf8398680e8
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,7c142cf63193a1473d2e66489a9ae977
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,72632f0f9dd73dfee390c9b22eb56dd6


In [11]:
customer_orders = order_financials.merge(
    order_customer[
        ["order_id", "customer_unique_id"]
    ],
    on="order_id",
    how="inner"
)

print("Customer-order financial data:")
print(customer_orders.head())

print("\nShape:", customer_orders.shape)

Customer-order financial data:
                           order_id  item_total  freight_total  total_payment  \
0  00010242fe8c5a6d1ba2dd792cb16214       58.90          13.29          72.19   
1  00018f77f2f0320c557190d7a144bdd3      239.90          19.93         259.83   
2  000229ec398224ef6ca0657da4fc703e      199.00          17.87         216.87   
3  00024acbcdf0a6daa1e931b038114c75       12.99          12.79          25.78   
4  00042b26cf59d7ce69dfabb4e55b4fd9      199.90          18.14         218.04   

                 customer_unique_id  
0  871766c5855e863f6eccc05f988b23cb  
1  eb28e67c4c0b83846050ddfb8a35d051  
2  3818d81c6709e39d06b2738a8d3a2474  
3  af861d436cfc08b2c2ddefd0ba074622  
4  64b576fb70d441e8f1b2d7d446e483c5  

Shape: (96478, 5)


In [12]:
customer_monetary = (
    customer_orders
    .groupby("customer_unique_id", as_index=False)
    .agg(
        Monetary=("total_payment", "sum")
    )
)

print("Final customer-level Monetary data:")
print(customer_monetary.head(10))

print("\nShape:", customer_monetary.shape)

Final customer-level Monetary data:
                 customer_unique_id  Monetary
0  0000366f3b9a7992bf8c76cfdf3221e2    141.90
1  0000b849f77a49e4a4ce2b2a4ca5be3f     27.19
2  0000f46a3911fa3c0805444483337064     86.22
3  0000f6ccb0745a6a4b88665a16c9f078     43.62
4  0004aac84e0df4da2b147fca70cf8255    196.89
5  0004bd2a26a76fe21f786e4fbd80607f    166.98
6  00050ab1314c0e55a6ca13cf7181fecf     35.38
7  00053a61a98854899e70ed204dd4bafe    419.18
8  0005e1862207bf6ccc02e4228effd9a0    150.12
9  0005ef4cd20d2893f0d9fbd94d3c0d97    129.76

Shape: (93358, 2)


In [13]:
print("===== FINAL VALIDATION =====")

print("\nColumns:")
print(customer_monetary.columns.tolist())

print("\nMissing values:")
print(customer_monetary.isnull().sum())

print(f"Total Unique Humans: {len(customer_monetary)}")

print("\nDuplicate customer IDs:")
print(customer_monetary["customer_unique_id"].duplicated().sum())

print("\nNegative Monetary values:")
print((customer_monetary["Monetary"] < 0).sum())

print("\nMonetary statistics:")
print(customer_monetary["Monetary"].describe())

===== FINAL VALIDATION =====

Columns:
['customer_unique_id', 'Monetary']

Missing values:
customer_unique_id    0
Monetary              0
dtype: int64
Total Unique Humans: 93358

Duplicate customer IDs:
0

Negative Monetary values:
0

Monetary statistics:
count    93358.000000
mean       165.197003
std        226.314012
min          0.000000
25%         63.052500
50%        107.780000
75%        182.557500
max      13664.080000
Name: Monetary, dtype: float64


In [14]:
print("Exporting Financial Metrics...")

# The Directory Defense
output_dir = "../data"
os.makedirs(output_dir, exist_ok=True) 

output_file = os.path.join(output_dir, "member2_monetary.csv")

# Save directly using the os path
customer_monetary.to_csv(output_file, index=False)

print(f"Success! Member 2 file saved to: {output_file}")
display(customer_monetary.head(3))

Exporting Financial Metrics...
Success! Member 2 file saved to: ../data\member2_monetary.csv


,customer_unique_id,Monetary
0,0000366f3b9a7992bf8c76cfdf3221e2,141.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,27.19
2,0000f46a3911fa3c0805444483337064,86.22
